In [ ]:
import pandas as pd
import sqlite3
import uuid
sample1_path= 'C:\\Users\\poste\\Downloads\\sample1.csv'
sample2_path= 'C:\\Users\\poste\\Downloads\\sample2.csv'
db_path= 'C:\\projectai\\ws2024-principles-of-ai-engineering\\database.db'

def combine_datasets(sample1_path, sample2_path, db_path, output_csv):
    # Charger les datasets CSV
    sample1 = pd.read_csv(sample1_path)
    sample2 = pd.read_csv(sample2_path)

    # Charger la base de données SQLite
    conn = sqlite3.connect(db_path)
    db_data = pd.read_sql_query("SELECT * FROM issues", conn)
    conn.close()

    # Harmoniser les colonnes des datasets CSV et de la base de données
    db_data.rename(columns={"title": "issue_title", "body": "issue_body"}, inplace=True)
    db_data = db_data[["id", "issue_title", "issue_body", "predicted_label", "corrected_label", "confidence"]]

    # Ajouter une colonne 'id' à sample1 et sample2 pour correspondre au format de la base de données
    sample1['id'] = [str(uuid.uuid4()) for _ in range(len(sample1))]
    sample2['id'] = [str(uuid.uuid4()) for _ in range(len(sample2))]

    # Uniformiser les colonnes pour tous les datasets
    common_columns = ["id", "issue_title", "issue_body", "issue_label"]
    sample1 = sample1[common_columns]
    sample2 = sample2[common_columns]
    db_data = db_data[["id", "issue_title", "issue_body", "predicted_label"]]

    # Renommer les colonnes pour correspondre au format final
    db_data.rename(columns={"predicted_label": "issue_label"}, inplace=True)

    # Combiner les trois datasets
    combined_data = pd.concat([sample1, sample2, db_data], ignore_index=True)

    # Supprimer les doublons basés sur issue_title et issue_body
    combined_data.drop_duplicates(subset=["issue_title", "issue_body"], inplace=True)

    # Exporter vers un fichier CSV
    combined_data.to_csv(output_csv, index=False)

    print(f"Combined data saved to {output_csv}")

# Exemple d'utilisation
combine_datasets('C:\\Users\\poste\\Downloads\\sample1.csv', 'C:\\Users\\poste\\Downloads\\sample2.csv', 'C:\\projectai\\ws2024-principles-of-ai-engineering\\database.db', "combined_issues.csv")

In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
data = pd.read_csv('C:\\Users\\poste\\Downloads\\combined_issues.csv')

# View the first 5 rows
data.head()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
data.info()


In [ ]:
# Filtrer l'issue où le titre est "Bug in login"
filtered_issue = data[data["issue_title"] == "Bug in login"]

# Afficher le résultat
print(filtered_issue)

In [ ]:
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import joblib

# Preprocess text function
def preprocess_text(text):
    if isinstance(text, str):  # Ensure the input is a string
        tokens = word_tokenize(text.lower())
        filtered_tokens = [token for token in tokens if token not in stopwords.words('english')]
        lemmatizer = WordNetLemmatizer()
        lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
        processed_text = ' '.join(lemmatized_tokens)
        return processed_text
    return ""  # Return an empty string for non-string inputs

# Read CSV and preprocess the data
data = pd.read_csv("C:\\Users\\poste\\Downloads\\combined_issues.csv")

# Drop rows where 'issue_title' or 'issue_body' is NaN
data = data.dropna(subset=['issue_title', 'issue_body'])

# Combine 'issue_title' and 'issue_body' into a single column
data['combined_text'] = data['issue_title'] + " " + data['issue_body']

# Preprocess the combined text
data['combined_text'] = data['combined_text'].apply(preprocess_text)

# Convert text to numerical data using TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data['combined_text'])
y = data['issue_label']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

# Predictions and evaluation
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)



# Specify the desired directory to save the files
save_path = r"C:\projectai\ws2024-principles-of-ai-engineering"

# Save the model and vectorizer
joblib.dump(rf, f"{save_path}\\final_random_forest_model.pkl")
joblib.dump(vectorizer, f"{save_path}\\final_tfidf_vectorizer.pkl")
